In [1]:
# # Универсальный RAG для анализа Python-репозиториев
#
# Настройте параметры ниже и запустите ноутбук.

# %%
# ============================================
# НАСТРОЙКИ (ИЗМЕНИТЕ ЗДЕСЬ)
# ============================================

# --- Источник кода ---
# Вариант 1: URL GitHub (рекомендуется для публичных репозиториев)
GITHUB_URL = "https://github.com/psf/requests.git"
REPO_PATH = "./temp_repo"  # Временная папка для клонирования

# Вариант 2: Локальный путь (закомментируйте GITHUB_URL и раскомментируйте следующую строку)
# REPO_PATH = "/YOU_PATH"
# GITHUB_URL = None

# --- Параметры анализа ---
GLOB_PATTERN = "**/*.py"       # Шаблон файлов для загрузки
CHUNK_SIZE = 2000              # Размер чанка (увеличьте до 4000, если контекст обрезается)
CHUNK_OVERLAP = 150
TOP_K_RESULTS = 5              # Количество фрагментов для поиска

# --- Параметры Ollama ---
EMBEDDING_MODEL = 'hf.co/Casual-Autopsy/snowflake-arctic-embed-l-v2.0-gguf:Q4_K_M'
LLM_MODEL = 'gemma3:4b'
OLLAMA_BASE_URL = 'http://127.0.0.1:11434'
LLM_TEMPERATURE = 0.7

# --- Режимы ---
RUN_DIAGNOSTICS = True         # Запустить детальную диагностику поиска?
USE_IMPROVED_PROMPT = True     # Использовать улучшенный аналитический промпт?

# ============================================



In [2]:
# ## 1. Подготовка репозитория

# %%
import os
import shutil
import subprocess
from pathlib import Path

def clone_github_repo(url: str, target_path: str) -> bool:
    """Клонирует GitHub репозиторий по URL"""
    print(f"Клонируем репозиторий {url} в {target_path}...")

    # Удаляем старую папку, если существует
    if os.path.exists(target_path):
        print(f"Удаляем старую папку {target_path}...")
        shutil.rmtree(target_path)

    # Клонируем репозиторий
    result = subprocess.run(
        ["git", "clone", url, target_path],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(f"Ошибка клонирования: {result.stderr}")
        return False

    print("Клонирование успешно завершено")
    return True

# Определяем источник кода
if GITHUB_URL:
    # Клонируем с GitHub
    if not clone_github_repo(GITHUB_URL, REPO_PATH):
        raise Exception(f"Не удалось клонировать репозиторий {GITHUB_URL}")

    # Специальная обработка для requests (известная проблема с часовыми поясами)
    if "requests" in GITHUB_URL:
        print("Применяем специальную настройку для requests...")
        subprocess.run(
            ["git", "config", "--global", "fetch.fsck.badTimezone", "ignore"],
            capture_output=True
        )
else:
    # Используем локальный путь
    print(f"Используем локальный репозиторий: {REPO_PATH}")

# Проверяем, что путь существует
if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(f"Путь {REPO_PATH} не найден. Проверьте настройки.")

print(f"Репозиторий готов к анализу: {REPO_PATH}")


Клонируем репозиторий https://github.com/psf/requests.git в ./temp_repo...
Клонирование успешно завершено
Применяем специальную настройку для requests...
Репозиторий готов к анализу: ./temp_repo


In [3]:
# ## 2. Загрузка Python файлов

# %%
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

print(f"Загружаем файлы по шаблону: {GLOB_PATTERN}")

loader = GenericLoader.from_filesystem(
    REPO_PATH,
    glob=GLOB_PATTERN,
    parser=LanguageParser(language="python")
)

docs = loader.load()

print(f"Загружено документов: {len(docs)}")
print(f"Суммарное количество символов: {sum(len(doc.page_content) for doc in docs):,}")


Загружаем файлы по шаблону: **/*.py
Загружено документов: 242
Суммарное количество символов: 384,635


In [4]:
# ## 3. Анализ загруженных документов

# %%
from collections import Counter

# Подсчёт файлов по типам (venv, site-packages, __pycache__)
files = [doc.metadata.get("source") for doc in docs]
counter = Counter()

for f in files:
    if f:
        if "site-packages" in f:
            counter["site-packages"] += 1
        elif "venv" in f:
            counter["venv"] += 1
        elif "__pycache__" in f:
            counter["pycache"] += 1

print("Распределение файлов:")
if counter:
    for key, value in counter.items():
        print(f"  {key}: {value}")
else:
    print("  Нет файлов из venv/site-packages/__pycache__")

# %%
lengths = [len(doc.page_content) for doc in docs]

print(f"Максимальная длина документа: {max(lengths):,} символов")
print(f"Средняя длина документа: {sum(lengths)/len(lengths):.2f} символов")
print(f"Медианная длина документа: {sorted(lengths)[len(lengths)//2]:,} символов")


Распределение файлов:
  Нет файлов из venv/site-packages/__pycache__
Максимальная длина документа: 79,236 символов
Средняя длина документа: 1589.40 символов
Медианная длина документа: 482 символов


In [6]:
# ## 4. Чистка кода и разбиение на чанки

# %%
def clean_code(text: str) -> str:
    """
    Обрезает слишком длинные строки (например, гигантские промпты),
    чтобы не перегружать векторную БД.
    """
    MAX_STRING_LENGTH = 1000

    if len(text) > 3000 and ("SYSTEM_PROMPT" in text or "prompt" in text.lower()):
        return text[:MAX_STRING_LENGTH] + "\n# ... truncated ..."

    return text

for doc in docs:
    doc.page_content = clean_code(doc.page_content)

# %%
long_docs = [d for d in docs if len(d.page_content) > 3000]
print(f"После очистки длинных документов осталось: {len(long_docs)}")

# %%
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

splits = []

for doc in docs:
    if len(doc.page_content) < CHUNK_SIZE * 3:
        splits.append(doc)
    else:
        splits.extend(splitter.split_documents([doc]))

print(f"Итоговое количество чанков: {len(splits)}")

# %%
# Статистика по чанкам
chunk_sizes = [len(s.page_content) for s in splits]
print(f"Размеры чанков:")
print(f"  Минимум: {min(chunk_sizes)} символов")
print(f"  Максимум: {max(chunk_sizes)} символов")
print(f"  Среднее: {sum(chunk_sizes)/len(chunk_sizes):.2f} символов")


После очистки длинных документов осталось: 20
Итоговое количество чанков: 356
Размеры чанков:
  Минимум: 0 символов
  Максимум: 5303 символов
  Среднее: 1080.72 символов


In [7]:
# ## 5. Создание эмбеддингов и векторного хранилища

# %%
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)

print(f"Используем модель эмбеддингов: {EMBEDDING_MODEL}")

# %%
from langchain_community.vectorstores import Chroma

persist_directory = "./chroma_db_temp"

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory,
)

print(f"Векторное хранилище создано: {persist_directory}")

# %%
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RESULTS}
)

print(f"Ретривер настроен: поиск {TOP_K_RESULTS} наиболее релевантных чанков")


Используем модель эмбеддингов: hf.co/Casual-Autopsy/snowflake-arctic-embed-l-v2.0-gguf:Q4_K_M
Векторное хранилище создано: ./chroma_db_temp
Ретривер настроен: поиск 5 наиболее релевантных чанков


In [8]:
# ## 6. Диагностика (опционально)

# %%
if RUN_DIAGNOSTICS:
    def inspect_retrieval(query: str):
        """Показывает, какие документы нашёл ретривер и их содержимое"""
        print(f"\n=== Диагностика по запросу: '{query}' ===\n")

        retrieved_docs = retriever.invoke(query)

        print(f"Найдено документов: {len(retrieved_docs)}\n")

        for i, doc in enumerate(retrieved_docs, 1):
            print(f"--- Документ {i} ---")
            print(f"Источник: {doc.metadata.get('source', 'unknown')}")
            print(f"Длина: {len(doc.page_content)} символов")
            print(f"Содержимое (первые 500 символов):\n{doc.page_content[:500]}")
            print("-" * 50)

    # Проверяем несколько запросов
    inspect_retrieval("как отправить POST запрос?")
    inspect_retrieval("класс Session")

    # Показываем структуру репозитория
    all_sources = set(doc.metadata.get('source') for doc in docs if doc.metadata.get('source'))
    print(f"\nВсего загружено файлов: {len(all_sources)}")
    print("\nПримеры загруженных файлов:")
    for source in sorted(list(all_sources))[:10]:
        print(f"  - {source}")



=== Диагностика по запросу: 'как отправить POST запрос?' ===

Найдено документов: 5

--- Документ 1 ---
Источник: temp_repo/src/requests/api.py
Длина: 595 символов
Содержимое (первые 500 символов):
def post(url, data=None, json=None, **kwargs):
    r"""Sends a POST request.

    :param url: URL for the new :class:`Request` object.
    :param data: (optional) Dictionary, list of tuples, bytes, or file-like
        object to send in the body of the :class:`Request`.
    :param json: (optional) A JSON serializable Python object to send in the body of the :class:`Request`.
    :param \*\*kwargs: Optional arguments that ``request`` takes.
    :return: :class:`Response <Response>` object
    :rt
--------------------------------------------------
--- Документ 2 ---
Источник: temp_repo/src/requests/api.py
Длина: 570 символов
Содержимое (первые 500 символов):
def put(url, data=None, **kwargs):
    r"""Sends a PUT request.

    :param url: URL for the new :class:`Request` object.
    :param dat

In [9]:
# ## 7. Настройка LLM и RAG цепочки

# %%
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=LLM_TEMPERATURE
)

print(f"Используем LLM: {LLM_MODEL}")

# %%
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    """Форматирует найденные документы в единый текст"""
    return "\n\n".join(doc.page_content for doc in docs)

if USE_IMPROVED_PROMPT:
    print("Используем УЛУЧШЕННЫЙ (аналитический) промпт")
    SYSTEM_PROMPT = """
    Ты — эксперт по анализу Python кода. Твоя задача — анализировать предоставленный код и давать полные ответы.

    Правила:
    1. Внимательно анализируй ВЕСЬ предоставленный контекст (классы, функции, импорты, комментарии, названия файлов).
    2. Если информации недостаточно, сделай обоснованные выводы из того, что есть (например, по импортам и настройкам).
    3. Структурируй ответ: основная функциональность, ключевые компоненты, взаимодействие с другими частями системы.
    4. Не говори "нет контекста" или "я не знаю", если можно сделать выводы.
    5. Отвечай на русском языке, даже если код на английском.
    """
else:
    print("Используем СТАНДАРТНЫЙ промпт")
    SYSTEM_PROMPT = """
    Ты помощник, который отвечает на вопросы по коду. Используй только предоставленный контекст.
    Если не знаешь ответа, скажи честно. Будь лаконичен. Отвечай на русском языке.
    """

HUMAN_PROMPT = "Контекст:\n{context}\n\nВопрос: {question}"

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", HUMAN_PROMPT),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG цепочка готова к работе")


Используем LLM: gemma3:4b
Используем УЛУЧШЕННЫЙ (аналитический) промпт
RAG цепочка готова к работе


In [10]:
# ## 8. Тестирование RAG системы

# %%
# Примеры вопросов для библиотеки requests
test_queries = [
    "как отправить GET запрос с параметрами?",
    "что делает класс Session?",
    "как обрабатываются Cookies?",
    "какие есть способы аутентификации?",
]

for query in test_queries:
    print("\n" + "=" * 60)
    print(f"Вопрос: {query}")
    print("=" * 60)

    try:
        response = rag_chain.invoke(query)
        print(f"Ответ:\n{response}")
    except Exception as e:
        print(f"Ошибка при выполнении запроса: {e}")

    print("-" * 60)


Вопрос: как отправить GET запрос с параметрами?
Ответ:
На основе предоставленного кода можно отправить GET запрос с параметрами следующим образом:

1.  **Использование функции `get()`**:  Функция `get(url, params=None, **kwargs)` предназначена для отправки GET запросов.  `params` - это аргумент, который принимает словарь, список кортежей или байты, которые будут добавлены в строку запроса (query string). `**kwargs` позволяет передавать другие аргументы, которые принимает функция `request()`.

2.  **Пример использования**:
    ```python
    import requests

    url = "http://www.example.com"
    params = {"key1": "value1", "key2": "value2"}  # Словарь параметров

    response = requests.get(url, params=params)

    print(response.url) # Выведет URL с параметрами: http://www.example.com/?key1=value1&key2=value2
    ```

    В этом примере `requests.get()` вызывается с указанием URL и словаря `params`.  `requests` автоматически преобразует словарь в строку запроса и добавляет его к URL.


In [11]:
# ## 9. Интерактивный режим

# %%
# Раскомментируйте для интерактивного режима

while True:
    print("\n" + "=" * 60)
    query = input("Введите вопрос (или 'exit' для выхода): ").strip()

    if query.lower() in ['exit', 'quit', 'q']:
        print("До свидания!")
        break

    if not query:
        print("Пожалуйста, введите вопрос.")
        continue

    print("\nОбработка запроса...")
    try:
        response = rag_chain.invoke(query)
        print(f"\nОтвет:\n{response}")
    except Exception as e:
        print(f"Ошибка: {e}")




Обработка запроса...

Ответ:
На основе предоставленного кода, метод `post` в данном контексте предназначен для отправки HTTP POST запроса.  Рассмотрим его работу по частям:

**Основная функциональность:**

Метод `post` является оберткой над функцией `request`.  Он принимает URL, данные, JSON и другие опции, которые затем передаются в функцию `request` для создания и отправки запроса.  В частности, он обрабатывает некоторые специфические сценарии, связанные с перенаправлениями (301, 302, 307) после отправки POST запроса,  чтобы имитировать поведение браузеров.

**Ключевые компоненты:**

1.  **Обработка перенаправлений:** Код содержит несколько проверок, которые изменяют метод запроса в зависимости от ответа сервера (статус-код).  Если сервер возвращает 302 (Found) или 307 (Temporary Redirect) после отправки POST запроса, метод запроса переключается на GET.  Это делается для соответствия поведению браузеров, которые автоматически перенаправляют GET запросы после получения POST запроса.